In [1]:
import json
import requests
import os
import polars as pl
import pandas as pd
from tqdm.notebook import tqdm

## 1. Загрузка

In [2]:
API_URL = "http://127.0.0.1:8000/"
PWD = os.getcwd()
INPUT_FILENAME = "input_10.json"
PWD_INPUT_FILE = PWD + "/../tests/" + INPUT_FILENAME

In [3]:
with open(PWD_INPUT_FILE, "r") as file:
    input_data = json.load(file)
input_data

{'days': ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday'],
 'default_count': 2,
 'groups': ['A-01-30',
  'A-02-30',
  'A-03-30',
  'A-04-30',
  'A-05-30',
  'A-06-30',
  'A-07-30',
  'A-08-30',
  'A-09-30',
  'A-10-30'],
 'rooms': ['m200',
  'm201',
  'm202',
  'm203',
  'm204',
  'm300',
  'm301',
  'm302',
  'm303',
  'm304'],
 'subjects': ['math', 'physics', 'english', 'IT', 'economic'],
 'teachers': ['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10'],
 'subject_teachers': {'math': ['T1', 'T2'],
  'physics': ['T3'],
  'english': ['T4', 'T5'],
  'IT': ['T6', 'T7'],
  'economic': ['T8', 'T9', 'T10']},
 'times': ['9:20', '11:10', '13:45', '15:35'],
 'subject_count': {'A-01-30': {'math': 2, 'IT': 1},
  'A-02-30': {'physics': 1},
  'A-03-30': {'english': 2},
  'A-04-30': {'economic': 1},
  'A-05-30': {'math': 1},
  'A-06-30': {'IT': 1},
  'A-07-30': {'english': 1},
  'A-08-30': {'math': 1},
  'A-09-30': {'economic': 1},
  'A-10-30': {'physics': 1}}}

In [4]:
def hello():
    try:
        resp = requests.get(API_URL + '') #json={"x1": x1, "x2": x2}
        resp.raise_for_status()
        data = resp.json()
        return data["result"]
    except Exception as e:
        return ("Error", str(e))

def solve_pulp():
    try:
        with open(PWD_INPUT_FILE, "r") as file:
            inp = json.load(file)
        resp = requests.post(API_URL + 'solve_pulp_2/', json=inp) #json={"x1": x1, "x2": x2}
        resp.raise_for_status()
        data = resp.json()
        return data["result"]
    except Exception as e:
        return e

In [5]:
hello()

'Hello'

In [6]:
mas = solve_pulp()
mas

[['group', 'day', 'time', 'subject', 'room', 'teacher'],
 ['A-01-30', 'Monday', '9:20', 'physics', 'm304', 'T3'],
 ['A-01-30', 'Monday', '11:10', 'english', 'm304', 'T4'],
 ['A-01-30', 'Monday', '13:45', 'math', 'm304', 'T1'],
 ['A-01-30', 'Thursday', '11:10', 'economic', 'm204', 'T8'],
 ['A-01-30', 'Thursday', '13:45', 'IT', 'm202', 'T7'],
 ['A-01-30', 'Thursday', '15:35', 'physics', 'm302', 'T3'],
 ['A-01-30', 'Friday', '11:10', 'english', 'm204', 'T4'],
 ['A-01-30', 'Friday', '13:45', 'economic', 'm300', 'T8'],
 ['A-01-30', 'Friday', '15:35', 'math', 'm303', 'T1'],
 ['A-02-30', 'Monday', '9:20', 'math', 'm301', 'T2'],
 ['A-02-30', 'Monday', '11:10', 'physics', 'm303', 'T3'],
 ['A-02-30', 'Monday', '13:45', 'english', 'm201', 'T4'],
 ['A-02-30', 'Monday', '15:35', 'english', 'm201', 'T5'],
 ['A-02-30', 'Tuesday', '9:20', 'IT', 'm200', 'T7'],
 ['A-02-30', 'Tuesday', '11:10', 'IT', 'm304', 'T7'],
 ['A-02-30', 'Tuesday', '13:45', 'economic', 'm204', 'T10'],
 ['A-02-30', 'Tuesday', '15:3

In [7]:
data_pl = pl.DataFrame(mas[1:], schema=mas[0], orient="row")
data_pd = pd.DataFrame(mas[1:], columns=mas[0])

In [8]:
data_pd

,group,day,time,subject,room,teacher
0,A-01-30,Monday,9:20,physics,m304,T3
1,A-01-30,Monday,11:10,english,m304,T4
2,A-01-30,Monday,13:45,math,m304,T1
3,A-01-30,Thursday,11:10,economic,m204,T8
4,A-01-30,Thursday,13:45,IT,m202,T7
...,...,...,...,...,...,...
86,A-10-30,Thursday,15:35,economic,m304,T8
87,A-10-30,Friday,9:20,physics,m203,T3
88,A-10-30,Friday,11:10,IT,m203,T7
89,A-10-30,Friday,13:45,IT,m203,T7


In [9]:
data_pl

group,day,time,subject,room,teacher
str,str,str,str,str,str
"""A-01-30""","""Monday""","""9:20""","""physics""","""m304""","""T3"""
"""A-01-30""","""Monday""","""11:10""","""english""","""m304""","""T4"""
"""A-01-30""","""Monday""","""13:45""","""math""","""m304""","""T1"""
"""A-01-30""","""Thursday""","""11:10""","""economic""","""m204""","""T8"""
"""A-01-30""","""Thursday""","""13:45""","""IT""","""m202""","""T7"""
…,…,…,…,…,…
"""A-10-30""","""Thursday""","""15:35""","""economic""","""m304""","""T8"""
"""A-10-30""","""Friday""","""9:20""","""physics""","""m203""","""T3"""
"""A-10-30""","""Friday""","""11:10""","""IT""","""m203""","""T7"""


## 2. Анализ условий

### 1) Для всех (group, subject) задать нужное количество занятий в неделю

In [10]:
input_data['default_count']

2

In [11]:
input_data['subject_count']

{'A-01-30': {'math': 2, 'IT': 1},
 'A-02-30': {'physics': 1},
 'A-03-30': {'english': 2},
 'A-04-30': {'economic': 1},
 'A-05-30': {'math': 1},
 'A-06-30': {'IT': 1},
 'A-07-30': {'english': 1},
 'A-08-30': {'math': 1},
 'A-09-30': {'economic': 1},
 'A-10-30': {'physics': 1}}

In [12]:
tmp = data_pl \
        .group_by("group", "subject") \
            .len()
tmp.head()

group,subject,len
str,str,u32
"""A-01-30""","""economic""",2
"""A-10-30""","""physics""",1
"""A-02-30""","""math""",2
"""A-10-30""","""IT""",2
"""A-05-30""","""math""",1


In [13]:
for group in tqdm(input_data['groups']):
    for subject in input_data['subjects']:
        try: 
            correct_count = input_data['subject_count'][group][subject]
        except KeyError: 
            correct_count = input_data['default_count']
        if not tmp.filter(
                    (pl.col("group") == group) & 
                    (pl.col("subject") == subject)
                )['len'].item() == correct_count:
            print(group, subject)

  0%|          | 0/10 [00:00<?, ?it/s]

### 2) Для каждого (day, time_slot, room) не более 1 занятия

В одной аудитории не может одновременно проводится более одного занатия

In [14]:
data_pl \
    .group_by("day", "time", "room") \
        .len() \
    .filter(pl.col("len") > 1)

day,time,room,len
str,str,str,u32


### 3) Для каждого (teacher, day, time_slot) не более 1 занятия

Преподаватель не может одновременно проводить более одного занятия.

In [15]:
data_pl \
    .group_by("teacher", "day", "time") \
        .len() \
    .filter(pl.col("len") > 1)

teacher,day,time,len
str,str,str,u32


### 4) Для каждой (group, day, time_slot) не более 1 занятия

У группы не может быть одновременно более одного занятия.

In [16]:
data_pl \
    .group_by("group", "day", "time") \
        .len() \
    .filter(pl.col("len") > 1)

group,day,time,len
str,str,str,u32


### 5) Окна для групп

Сначала переведём время в минуты.

In [18]:
minutes = data_pl \
    .with_columns(
        start_min=(
            pl.col("time")
            .str.split(":")
            .list.eval(pl.element().cast(pl.Int32))
            .list.first() * 60
            + pl.col("time")
            .str.split(":")
            .list.eval(pl.element().cast(pl.Int32))
            .list.last()
        )
    )
minutes.head()

group,day,time,subject,room,teacher,start_min
str,str,str,str,str,str,i32
"""A-01-30""","""Monday""","""9:20""","""physics""","""m304""","""T3""",560
"""A-01-30""","""Monday""","""11:10""","""english""","""m304""","""T4""",670
"""A-01-30""","""Monday""","""13:45""","""math""","""m304""","""T1""",825
"""A-01-30""","""Thursday""","""11:10""","""economic""","""m204""","""T8""",670
"""A-01-30""","""Thursday""","""13:45""","""IT""","""m202""","""T7""",825


In [19]:
LESSON = 90  # минут

windows = minutes \
    .sort(["group", "day", "start_min"]) \
    .with_columns(
        gap_min=(
            pl.col("start_min").shift(-1).over(["group", "day"])
            - (pl.col("start_min") + LESSON)
        )
    ) \
    .filter(pl.col("gap_min") > 90) \
    .select("group", "day", "start_min", "gap_min")


print(windows)


shape: (0, 4)
┌───────┬─────┬───────────┬─────────┐
│ group ┆ day ┆ start_min ┆ gap_min │
│ ---   ┆ --- ┆ ---       ┆ ---     │
│ str   ┆ str ┆ i32       ┆ i32     │
╞═══════╪═════╪═══════════╪═════════╡
└───────┴─────┴───────────┴─────────┘


### 6) Окна для преподавателей

In [20]:
LESSON = 90  # минут

windows = (minutes
    .sort(["teacher", "day", "start_min"])
    .with_columns(
        gap_min=(
            pl.col("start_min").shift(-1).over(["teacher", "day"])
            - (pl.col("start_min") + LESSON)
        )
    )
    .filter(pl.col("gap_min") > 90)
    .select("teacher", "day", "start_min", "gap_min")
)

print(windows)

shape: (0, 4)
┌─────────┬─────┬───────────┬─────────┐
│ teacher ┆ day ┆ start_min ┆ gap_min │
│ ---     ┆ --- ┆ ---       ┆ ---     │
│ str     ┆ str ┆ i32       ┆ i32     │
╞═════════╪═════╪═══════════╪═════════╡
└─────────┴─────┴───────────┴─────────┘


# 3. Доказательство 

![изображение.png](./img/ganta_10.png)